In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/abraham/uni/ikt453/project/v1

/home/abraham/uni/ikt453/project/v1


## The Team Dimension 
- seeded by
    - [x] data/samples/leaugeHiearchy.json
    - [x] data/samples_nbaapi/boxscores.json
- properties
    - team_name
    - team_alias
    - team_market
    - division_name
    - division_alias
    - conference_name
    - conference_alias
    - founded_in
    - venue_name
    - venue_capacity
    - venue_address
    - venue_city
    - venue_state
    - venue_zip

## The Team Roles Dimension
- seeded by
    - [x] data/samples/leaugeHiearchy.json
    - [x] data/samples_nbaapi/boxscores.json
- properties
    - mascot
    - sponsor
    - general_manager

## The Team Fact
- seeded by
    - [x] data/samples/leaugeHiearchy.json
    - [x] data/samples_nbaapi/boxscores.json
- properties
    - championships_won
    - championship_seasons
    - conference_titles
    - division_titles

In [5]:
from src.utils import disk
from src.utils import debug
from src.processing.ids import id_mapper, IDMapper

In [6]:
league_hiarchy = 'data/samples/leaugeHiearchy.json'
boxscores = 'data/samples_nbaapi/boxscores.json'

lh = disk.read_json(league_hiarchy)
bs = disk.read_json(boxscores)

In [12]:
from uuid import uuid4
from itertools import chain

def construct_team_dimension(lh: dict, id_mapper: IDMapper, preserve: tuple[str] = None) -> tuple[list, list, list, dict]:
    preserve = preserve or ()

    # ------------------------
    # Helper function to explode values on multiple delimiters and preserve certain values
    # ------------------------    
    def explode(output_key: str, key: str, obj: dict, **kwargs):
        split_on = (',', ' and ', '&')
        value = obj.get(key, '')
        if not value:
            return []
        
        if value in preserve or not isinstance(value, str):
            return [dict(**kwargs, **{output_key: value})]
        
        values: list[str] = [[value]]
        for splitter in split_on:
            values = chain.from_iterable(values)
            values = list(map(lambda v: v.split(splitter), values))
        
        values = chain.from_iterable(values)
        values = map(lambda v: v.strip(), values)
        values = filter(lambda v: v, values)
        return list(map(lambda v: dict(**kwargs, **{output_key: v}), values))

    # ------------------------
    # Main logic to construct teams, team roles, and team facts
    # ------------------------
    teams = []
    roles = []
    facts = []

    conferences = lh['conferences']
    for conference in conferences:
        c_id = id_mapper(
            namespace='conference',
            key=conference['alias'],
            provider='sportsradar',
            provider_id=conference['id'],
            custom_id=str(uuid4()),
        )

        for division in conference['divisions']:
            d_id = id_mapper(
                namespace='division',
                key=division['alias'],
                provider='sportsradar',
                provider_id=division['id'],
                custom_id=str(uuid4()),
            )

            for team in division['teams']:
                t_id = id_mapper(
                    namespace='team',
                    key=team['alias'],
                    provider='sportsradar',
                    provider_id=team['id'],
                    custom_id=str(uuid4()),
                )
                
                venue = team['venue']

                meta = dict(
                    team_id=t_id,
                    team_name=team['name'],
                    team_alias=team['alias'],
                )

                teams.append(dict(
                    **meta,
                    team_market=team['market'],
                    division_id=d_id,
                    division_name=division['name'],
                    division_alias=division['alias'],
                    conference_id=c_id,
                    conference_name=conference['name'],
                    conference_alias=conference['alias'],
                    founded_in=team['founded'],
                    venue_name=venue['name'],
                    venue_capacity=venue['capacity'],
                    venue_address=venue['address'],
                    venue_city=venue['city'],
                    venue_state=venue['state'],
                    venue_zip=venue['zip'],
                    venue_country=venue['country'],
                ))


                roles.extend(chain.from_iterable(map(
                    lambda role: explode('name', role, team, **meta, role=role), 
                    [
                        'mascot',
                        'sponsor',
                        'owner',
                        'general_manager',
                    ]
                )))
    
                facts.extend(chain.from_iterable(map(
                    lambda fact: explode('value', fact, team, **meta, fact=fact),
                    [
                        'championships_won',
                        'championship_seasons',
                        'playoff_appearances',
                        'conference_titles',
                        'division_titles',
                    ]
                )))


    return teams, roles, facts

In [13]:
teams, team_roles, team_facts = construct_team_dimension(lh, id_mapper, preserve=("Kroenke Sports & Entertainment", ))

In [31]:
import pandas as pd

df_teams = pd.DataFrame(teams)
df_roles = pd.DataFrame(team_roles)
df_facts = pd.DataFrame(team_facts)

df_teams.iloc[0]

team_id             01b0bf34-c492-4b1f-b93e-9bfbe849abcf
team_name                                        Wizards
team_alias                                           WAS
team_market                                   Washington
division_id         59b168e6-8c4c-4c42-8fc1-784c057212c0
division_name                                  Southeast
division_alias                                 SOUTHEAST
conference_id       2a2cb4c5-103b-4f23-b620-576b2e7200fa
conference_name                       EASTERN CONFERENCE
conference_alias                                 EASTERN
founded_in                                          1961
venue_name                             Capital One Arena
venue_capacity                                     20356
venue_address                            601 F Street NW
venue_city                                    Washington
venue_state                                           DC
venue_zip                                          20004
venue_country                  

In [36]:
disk.write_json('data/_v2_dimensions/nba_teams.json', teams)

In [32]:
df_roles.head()

,team_id,team_name,team_alias,role,name
0,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,mascot,G-Wiz
1,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,sponsor,Robinhood
2,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,owner,Ted Leonsis
3,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,general_manager,Will Dawkins
4,e760d6ee-350f-45cb-891d-bdeed1aa9f98,Hornets,CHA,mascot,Hugo the Hornet


In [33]:
df_facts.head()

,team_id,team_name,team_alias,fact,value
0,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,championships_won,1
1,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,championship_seasons,1978
2,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,playoff_appearances,30
3,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,conference_titles,4
4,01b0bf34-c492-4b1f-b93e-9bfbe849abcf,Wizards,WAS,division_titles,8


In [18]:
debug.prettyprint(id_mapper.ids)

{
    "conference": {
        "EASTERN": {
            "sportsradar": "3960cfac-7361-4b30-bc25-8d393de6f62f",
            "custom": "2a2cb4c5-103b-4f23-b620-576b2e7200fa"
        },
        "WESTERN": {
            "sportsradar": "7fe7e212-de01-4f8f-a31d-b9f0a95731e3",
            "custom": "a8256a3b-77c8-48ea-8f28-d6167ab707af"
        }
    },
    "division": {
        "SOUTHEAST": {
            "sportsradar": "54dc7348-c1d2-40d8-88b3-c4c0138e085d",
            "custom": "59b168e6-8c4c-4c42-8fc1-784c057212c0"
        },
        "ATLANTIC": {
            "sportsradar": "582d6502-9a93-4a8d-8785-69374d732875",
            "custom": "ea2534c6-96bb-4bee-9dae-a284302212cc"
        },
        "CENTRAL": {
            "sportsradar": "f3aaf23a-1ceb-46ef-8fef-9403692e801b",
            "custom": "9cab06cf-09fb-481c-a233-41c7f5f2e758"
        },
        "NORTHWEST": {
            "sportsradar": "12bf14ba-eb16-4c6f-8275-e801b6947c1e",
            "custom": "a79125cd-c9a7-4a30-9a71-cc5937e43dbb"
